In [1]:
"""
===============================================================================
KAGGLE NOTEBOOK 1: HMDA Data Download & Setup
===============================================================================
Run this FIRST on Kaggle (CPU only - no GPU needed)

What this notebook does:
1. Downloads the HMDA (Home Mortgage Disclosure Act) dataset
2. Verifies the download and shows basic info
3. Saves data to /kaggle/working/ for subsequent notebooks

HMDA Dataset Info:
- Source: Consumer Financial Protection Bureau (CFPB)
- Contains: Mortgage application data with demographics
- Size: ~20M+ records annually (we use a subset)
- Key features: Loan amount, income, DTI, LTV, demographics, action taken
- Target: action_taken (1=originated, 3=denied)

KAGGLE SETUP:
- Create new notebook on kaggle.com
- Set Accelerator: None (CPU)
- Runtime: ~5-10 minutes
===============================================================================
"""

'\n===============================================================================\nKAGGLE NOTEBOOK 1: HMDA Data Download & Setup\n===============================================================================\nRun this FIRST on Kaggle (CPU only - no GPU needed)\n\nWhat this notebook does:\n1. Downloads the HMDA (Home Mortgage Disclosure Act) dataset\n2. Verifies the download and shows basic info\n3. Saves data to /kaggle/working/ for subsequent notebooks\n\nHMDA Dataset Info:\n- Source: Consumer Financial Protection Bureau (CFPB)\n- Contains: Mortgage application data with demographics\n- Size: ~20M+ records annually (we use a subset)\n- Key features: Loan amount, income, DTI, LTV, demographics, action taken\n- Target: action_taken (1=originated, 3=denied)\n\nKAGGLE SETUP:\n- Create new notebook on kaggle.com\n- Set Accelerator: None (CPU)\n- Runtime: ~5-10 minutes\n===============================================================================\n'

### Install Requirements & Imports

In [2]:
import pandas as pd
import numpy as np
import os
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set paths for Kaggle
WORKING_DIR = Path("/kaggle/working")
DATA_DIR = WORKING_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = WORKING_DIR / "results"

# Create directories
for d in [DATA_DIR, RAW_DIR, PROCESSED_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("KAGGLE NOTEBOOK 1: HMDA Data Download & Setup")
print("=" * 70)
print(f"Working directory: {WORKING_DIR}")
print(f"Data will be saved to: {RAW_DIR}")

KAGGLE NOTEBOOK 1: HMDA Data Download & Setup
Working directory: /kaggle/working
Data will be saved to: /kaggle/working/data/raw


### Download HMDA Dataset

In [3]:
"""
Download from Kaggle Dataset (RECOMMENDED)

The HMDA dataset is available as a Kaggle dataset.
Add this dataset to your notebook:
1. Click "Add Input" in the right panel
2. Search for "HMDA" or "Home Mortgage Disclosure Act"
3. Select the CFPB HMDA dataset (2025)

After adding, the data will be at: /kaggle/input/hmda-...
"""

# Try to find HMDA data in Kaggle input
KAGGLE_INPUT = Path("/kaggle/input")

print("--- Scanning Kaggle Input Directory ---")
if KAGGLE_INPUT.exists():
    # os.walk forces Python to look deep inside all nested folders
    for root, dirs, files in os.walk(KAGGLE_INPUT):
        for file in files:
            if file.endswith(('.csv', '.parquet')):
                full_path = Path(root) / file
                print(f"Found File: {file}")
                print(f"Absolute Path: {full_path}\n")
else:
    print("⚠️ /kaggle/input directory does not exist!")

--- Scanning Kaggle Input Directory ---
Found File: HMDA_year_2025.csv
Absolute Path: /kaggle/input/datasets/jaykalbi/2025-hmda-nationwide-mortgage-lending-dataset/HMDA_year_2025.csv



In [4]:
# Defining main working project directory
PROJECT_DIR = Path("/kaggle/working")

# Create a 'data' or 'raw' subdirectory structure
RAW_DIR = PROJECT_DIR / "data_raw"

# Safely generate the physical folders on disk if they don't exist yet
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Output directory initialized at: {RAW_DIR}")

✅ Output directory initialized at: /kaggle/working/data_raw


### Load Data

In [5]:
# 1. DEFINE THE FUNCTION FIRST
def download_hmda_data(source="kaggle", sample_size=500000):
    """
    Loads HMDA data from Kaggle input or falls back to a sample.
    """
    # Note: Check this path. Your directory list cell showed empty output for 'hmda'
    kaggle_path = Path("/kaggle/input/datasets/jaykalbi/2025-hmda-nationwide-mortgage-lending-dataset/HMDA_year_2025.csv")
    
    if source == "kaggle":
        if not kaggle_path.exists():
            raise FileNotFoundError(f"Kaggle file not found at: {kaggle_path}")
        print(f"Reading data from {kaggle_path}...")
        return pd.read_csv(kaggle_path, nrows=sample_size, low_memory=False)
        
    elif source == "sample":
        print(f"Generating random sample dataframe of size {sample_size}...")
        mock_data = {
            'action_taken': np.random.choice([1, 2, 3], size=sample_size),
            'loan_amount': np.random.randint(50000, 500000, size=sample_size),
            'applicant_income': np.random.randint(30000, 200000, size=sample_size)
        }
        return pd.DataFrame(mock_data)
    else:
        raise ValueError("Invalid source. Choose 'kaggle' or 'sample'.")

# 2. RUN THE EXECUTION CODE SECOND
try:
    df = download_hmda_data(source="kaggle")
except Exception as e:
    print(f"Kaggle input not available: {e}")
    print("Using sample data for demonstration...")
    df = download_hmda_data(source="sample", sample_size=500000)

# Save raw data (Ensure RAW_DIR is defined above this cell)
df.to_parquet(RAW_DIR / "hmda_raw.parquet", index=False)
print(f"\nSaved raw data to {RAW_DIR}/hmda_raw.parquet")

Reading data from /kaggle/input/datasets/jaykalbi/2025-hmda-nationwide-mortgage-lending-dataset/HMDA_year_2025.csv...

Saved raw data to /kaggle/working/data_raw/hmda_raw.parquet


### Data Verification & Basic Info

In [6]:
print("\n" + "=" * 70)
print("DATA VERIFICATION")
print("=" * 70)

print(f"\nShape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumn names:\n{list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values per column:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nDuplicated rows: {df.duplicated().sum():,}")


DATA VERIFICATION

Shape: 500,000 rows x 99 columns

Column names:
['activity_year', 'lei', 'derived_msa-md', 'state_code', 'county_code', 'census_tract', 'conforming_loan_limit', 'derived_loan_product_type', 'derived_dwelling_category', 'derived_ethnicity', 'derived_race', 'derived_sex', 'action_taken', 'purchaser_type', 'preapproval', 'loan_type', 'loan_purpose', 'lien_status', 'reverse_mortgage', 'open-end_line_of_credit', 'business_or_commercial_purpose', 'loan_amount', 'loan_to_value_ratio', 'interest_rate', 'rate_spread', 'hoepa_status', 'total_loan_costs', 'total_points_and_fees', 'origination_charges', 'discount_points', 'lender_credits', 'loan_term', 'prepayment_penalty_term', 'intro_rate_period', 'negative_amortization', 'interest_only_payment', 'balloon_payment', 'other_nonamortizing_features', 'property_value', 'construction_method', 'occupancy_type', 'manufactured_home_secured_property_type', 'manufactured_home_land_property_interest', 'total_units', 'multifamily_affordab

### Target Variable Analysis

In [7]:
print("\n" + "=" * 70)
print("TARGET VARIABLE ANALYSIS")
print("=" * 70)

# HMDA action_taken codes:
action_codes = {
    1: 'Loan originated',
    2: 'Application approved but not accepted',
    3: 'Application denied',
    4: 'Application withdrawn by applicant',
    5: 'File closed for incompleteness',
    6: 'Purchased loan',
    7: 'Preapproval request denied',
    8: 'Preapproval request approved but not accepted'
}

print("\nAction Taken Distribution:")
action_counts = df['action_taken'].value_counts().sort_index()
for code, count in action_counts.items():
    desc = action_codes.get(code, 'Unknown')
    pct = count / len(df) * 100
    print(f"  {code}: {desc:45s} | {count:>10,} ({pct:5.1f}%)")

# Focus on originated (1) vs denied (3)
print("\n--- Binary Target (Originated vs Denied) ---")
df_binary = df[df['action_taken'].isin([1, 3])].copy()
df_binary['default'] = (df_binary['action_taken'] == 3).astype(int)
print(f"Records with binary outcome: {len(df_binary):,}")
print(f"Denial rate: {df_binary['default'].mean():.2%}")

# Save metadata
metadata = {
    'dataset': 'HMDA',
    'total_records': int(len(df)),
    'binary_records': int(len(df_binary)),
    'denial_rate': float(df_binary['default'].mean()),
    'columns': list(df.columns),
    'action_taken_distribution': {str(k): int(v) for k, v in action_counts.items()},
    'data_year': 2022,
    'download_date': pd.Timestamp.now().strftime('%Y-%m-%d')
}

with open(RESULTS_DIR / "download_metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\nSaved metadata to {RESULTS_DIR}/download_metadata.json")


TARGET VARIABLE ANALYSIS

Action Taken Distribution:
  1: Loan originated                               |    383,391 ( 76.7%)
  2: Application approved but not accepted         |     10,722 (  2.1%)
  3: Application denied                            |      1,541 (  0.3%)
  4: Application withdrawn by applicant            |      1,958 (  0.4%)
  5: File closed for incompleteness                |      1,250 (  0.2%)
  6: Purchased loan                                |    100,939 ( 20.2%)
  7: Preapproval request denied                    |         14 (  0.0%)
  8: Preapproval request approved but not accepted |        185 (  0.0%)

--- Binary Target (Originated vs Denied) ---
Records with binary outcome: 384,932
Denial rate: 0.40%

Saved metadata to /kaggle/working/results/download_metadata.json


### Quick Profile of Key Features

In [8]:
print("\n" + "=" * 70)
print("QUICK FEATURE PROFILE")
print("=" * 70)

# Numeric features
numeric_cols = ['loan_amount', 'income', 'loan_to_value_ratio', 
                'debt_to_income_ratio', 'property_value', 'loan_term']

print("\nNumeric Features Summary:")
for col in numeric_cols:
    if col in df.columns:
        # --- FIX: Convert column to numeric, forcing strings/text to NaN ---
        series = pd.to_numeric(df[col], errors='coerce')
        
        print(f"\n{col}:")
        print(f"  Mean: {series.mean():,.2f}")
        print(f"  Median: {series.median():,.2f}")
        print(f"  Min: {series.min():,.2f}")
        print(f"  Max: {series.max():,.2f}")
        print(f"  Missing/Exempt: {series.isnull().sum():,} ({series.isnull().mean():.1%})")

# Categorical features
cat_cols = ['loan_type', 'loan_purpose', 'lien_status', 'occupancy_type',
            'applicant_sex', 'applicant_ethnicity']

print("\nCategorical Features - Top Values:")
for col in cat_cols:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts().head(5).to_string())



QUICK FEATURE PROFILE

Numeric Features Summary:

loan_amount:
  Mean: 327,830.04
  Median: 275,000.00
  Min: 5,000.00
  Max: 119,605,000.00
  Missing/Exempt: 0 (0.0%)

income:
  Mean: 154.57
  Median: 119.00
  Min: -416.00
  Max: 85,184.00
  Missing/Exempt: 51,316 (10.3%)

loan_to_value_ratio:
  Mean: 80.03
  Median: 80.00
  Min: 2.09
  Max: 513.45
  Missing/Exempt: 107,092 (21.4%)

debt_to_income_ratio:
  Mean: 43.56
  Median: 44.00
  Min: 36.00
  Max: 49.00
  Missing/Exempt: 300,024 (60.0%)

property_value:
  Mean: 527,638.71
  Median: 425,000.00
  Min: 25,000.00
  Max: 241,805,000.00
  Missing/Exempt: 5,850 (1.2%)

loan_term:
  Mean: 333.62
  Median: 360.00
  Min: 1.00
  Max: 836.00
  Missing/Exempt: 1,511 (0.3%)

Categorical Features - Top Values:

loan_type:
loan_type
1    355154
2     72329
3     71446
4      1071

loan_purpose:
loan_purpose
1     222677
32    143536
31    129090
5       2686
4       1486

lien_status:
lien_status
1    400988
2     99012

occupancy_type:
occupa

### Save Cleaned Binary Dataset

In [9]:
# Save the binary-focused dataset for next notebook
df_binary.to_parquet(PROCESSED_DIR / "hmda_binary.parquet", index=False)
print(f"\nSaved binary dataset to {PROCESSED_DIR}/hmda_binary.parquet")

print("\n" + "=" * 70)
print("Dataset Download COMPLETE!")
print("=" * 70)
print(f"\nNext: Run Notebook 2 (EDA) on this dataset")
print(f"File: {PROCESSED_DIR}/hmda_binary.parquet")
print(f"Shape: {df_binary.shape}")


Saved binary dataset to /kaggle/working/data/processed/hmda_binary.parquet

Dataset Download COMPLETE!

Next: Run Notebook 2 (EDA) on this dataset
File: /kaggle/working/data/processed/hmda_binary.parquet
Shape: (384932, 100)
